# Test Claude connection through AgentRouter

This notebook tests configuration without displaying or storing the API key in notebook output. The Claude Code CLI test is the primary check because `agentrouter.org` may reject generic HTTP clients that do not resemble Claude Code.

Expected project configuration:

- Token source: `https://agentrouter.org/console/token`
- Base URL: `https://agentrouter.org`
- Model: `claude-opus-5`
- Local key file: project-root `.env` containing `AGENTROUTER_API_KEY=...`

In [1]:
from __future__ import annotations

import getpass
import json
import os
import subprocess
from pathlib import Path

import httpx

BASE_URL = "https://agentrouter.org"
MODEL = "claude-opus-5"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"
CLAUDE_CANDIDATES = [Path.home() / ".local/bin/claude", Path("claude")]
print({"project_root": str(PROJECT_ROOT), "base_url": BASE_URL, "model": MODEL})

{'project_root': '/Users/meisam/Documents/text-to-sql/semantic_text2sql_v4', 'base_url': 'https://agentrouter.org', 'model': 'claude-opus-5'}


In [2]:
def load_key() -> str:
    key = os.environ.get("AGENTROUTER_API_KEY", "").strip()
    if not key and ENV_FILE.is_file():
        for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
            if line.startswith("AGENTROUTER_API_KEY="):
                key = line.split("=", 1)[1].strip().strip("\"'")
                break
    if not key:
        key = getpass.getpass("AgentRouter API key (hidden): ").strip()
    if not key:
        raise RuntimeError("No AgentRouter key was provided.")
    return key


API_KEY = load_key()
print(
    {
        "key_loaded": True,
        "starts_with_sk": API_KEY.startswith("sk-"),
        "contains_whitespace": any(char.isspace() for char in API_KEY),
        "key_length": len(API_KEY),
    }
)

{'key_loaded': True, 'starts_with_sk': True, 'contains_whitespace': False, 'key_length': 51}


## Optional raw HTTP diagnostic

A `401 unauthorized_client` here can mean AgentRouter rejected a generic HTTP client. It does not prove that the same token will fail through Claude Code.

In [3]:
def raw_http_test() -> dict[str, object]:
    try:
        response = httpx.post(
            f"{BASE_URL}/v1/messages?beta=true",
            headers={
                "x-api-key": API_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": MODEL,
                "max_tokens": 20,
                "temperature": 0,
                "messages": [{"role": "user", "content": "Reply exactly: CLAUDE_OK"}],
            },
            timeout=30,
        )
        return {"status_code": response.status_code, "body": response.text[:1000]}
    except httpx.HTTPError as exc:
        return {"status_code": None, "error": f"{type(exc).__name__}: {exc}"}


raw_result = raw_http_test()
print(json.dumps(raw_result, indent=2))

{
  "status_code": 401,
  "body": "{\"error\":{\"message\":\"unauthorized client detected, contact support for assistance at https://discord.gg/aYq5B4RW3\"},\"message\":\"UNAUTHENTICATED\",\"success\":false,\"type\":\"unauthorized_client_error\"}"
}


## Primary Claude Code test

This invokes the official Claude Code binary in non-interactive mode, disables tools, limits the run to one turn, and stops after 120 seconds.

In [4]:
def find_claude() -> str:
    native = CLAUDE_CANDIDATES[0]
    if native.is_file():
        return str(native)
    return "claude"


def claude_code_test(timeout_seconds: int = 120) -> dict[str, object]:
    environment = os.environ.copy()
    environment.update(
        {
            "ANTHROPIC_BASE_URL": BASE_URL,
            "ANTHROPIC_AUTH_TOKEN": API_KEY,
            "ANTHROPIC_API_KEY": API_KEY,
            "CLAUDE_CODE_USE_AUTH_TOKEN": "true",
            "ANTHROPIC_MODEL": MODEL,
        }
    )
    command = [
        find_claude(),
        "-p",
        "--model",
        MODEL,
        "--tools",
        "",
        "--max-turns",
        "1",
        "--output-format",
        "json",
        "Reply exactly: CLAUDE_OK",
    ]
    try:
        completed = subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            env=environment,
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            check=False,
        )
    except FileNotFoundError:
        return {"connected": False, "error": "Claude Code is not installed."}
    except subprocess.TimeoutExpired as exc:
        return {
            "connected": False,
            "error": f"Claude Code returned no result within {timeout_seconds} seconds.",
            "stdout": (exc.stdout or "")[-2000:],
            "stderr": (exc.stderr or "")[-2000:],
        }
    stdout = completed.stdout.strip()
    try:
        payload = json.loads(stdout) if stdout else {}
    except json.JSONDecodeError:
        payload = {"raw_stdout": stdout[-2000:]}
    text = str(payload.get("result", ""))
    return {
        "connected": completed.returncode == 0 and "CLAUDE_OK" in text,
        "return_code": completed.returncode,
        "result": text,
        "terminal_reason": payload.get("terminal_reason"),
        "is_error": payload.get("is_error"),
        "usage": payload.get("usage"),
        "errors": payload.get("errors"),
        "stderr": completed.stderr[-2000:],
    }


claude_result = claude_code_test()
print(json.dumps(claude_result, indent=2))

{
  "connected": true,
  "return_code": 0,
  "result": "CLAUDE_OK",
  "terminal_reason": "completed",
  "is_error": false,
  "usage": {
    "input_tokens": 2,
    "cache_creation_input_tokens": 3385,
    "cache_read_input_tokens": 0,
    "output_tokens": 11,
    "server_tool_use": {
      "web_search_requests": 0,
      "web_fetch_requests": 0
    },
    "service_tier": "standard",
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 3385
    },
    "inference_geo": "",
    "iterations": [],
    "speed": "standard"
  },
  "errors": null,
  "stderr": "\u001bWarning: no stdin data received in 3s, proceeding without it. If piping from a slow command, redirect stdin explicitly: < /dev/null to skip, or wait longer.\u001b\n"
}


In [5]:
if claude_result.get("connected"):
    print("PASS: Claude returned the expected response.")
elif claude_result.get("terminal_reason") == "aborted_streaming":
    print(
        "FAIL: Claude Code started, but AgentRouter produced no model stream. "
        "Check token status, credit, and gateway availability."
    )
elif claude_result.get("error"):
    print(f"FAIL: {claude_result['error']}")
else:
    print("FAIL: Inspect the sanitized diagnostic above.")

PASS: Claude returned the expected response.
